# DATAFLOW 2026 - User Behavior Prediction
## Part 2: Feature Engineering

---

### Data Policy ? STRICT RULE (must never be violated)

| Rule | Detail |
|------|--------|
| `SequencePreprocessor.fit()` | **Only on `train_sequences`** ? never on val or test |
| `FeaturePipeline.fit()` | **Only on `train_sequences`** ? never on val or test |
| `TargetEncoder.fit()` | **Only on `Y_train`** ? never on val labels |
| `Y_val` | Transformed (not fitted) ? used only for offline evaluation in notebook 03 |
| `merge_train_val()` | **NEVER called** |
| Dense vocab mapping | Built from train tokens only; unknown val/test tokens map to 0 (padding) |

---

### Notebook Contents
1. Sequence Preprocessing (SequencePreprocessor ? dense vocab remapping + padding)
2. TF-IDF Features
3. N-gram Features
4. Statistical + Histogram Features
5. Full Feature Pipeline (for XGBoost / LightGBM)
6. Target Encoding

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data import DataLoader
from src.data.preprocessor import (
    SequencePreprocessor, 
    TargetEncoder, 
    extract_sequences_from_df
)
from src.features import (
    SequenceFeatureExtractor,
    StatisticalFeatureExtractor,
    FeaturePipeline
)
from src.utils import set_seed, load_config

# Create output directories
Path('../outputs/figures').mkdir(parents=True, exist_ok=True)
Path('../outputs/models').mkdir(parents=True, exist_ok=True)
Path('../outputs/submissions').mkdir(parents=True, exist_ok=True)

set_seed(42)
%matplotlib inline

In [ ]:
# Load data
config = load_config('../configs/config.yaml')
loader = DataLoader('../data')

X_train, Y_train = loader.load_train()
X_val, Y_val = loader.load_val()

train_ids, train_sequences = extract_sequences_from_df(X_train)
val_ids, val_sequences = extract_sequences_from_df(X_val)

print(f"Training sequences: {len(train_sequences)}")
print(f"Validation sequences: {len(val_sequences)}")

## 1. Sequence Preprocessing

In [ ]:
# Create sequence preprocessor
preprocessor = SequencePreprocessor(
    max_length=config['preprocessing']['max_sequence_length'],
    padding_strategy=config['preprocessing']['padding_strategy'],
    truncation_strategy=config['preprocessing']['truncation_strategy'],
)

# Fit and transform
X_train_padded = preprocessor.fit_transform(train_sequences)
X_val_padded = preprocessor.transform(val_sequences)

print(f"Padded train shape: {X_train_padded.shape}")
print(f"Padded val shape: {X_val_padded.shape}")
print(f"Vocabulary size: {preprocessor.vocab_size}")

In [ ]:
# Generate attention masks
train_masks = preprocessor.get_attention_mask(train_sequences)
val_masks = preprocessor.get_attention_mask(val_sequences)

print(f"Train mask shape: {train_masks.shape}")
print(f"Sample mask: {train_masks[0][:20]}...")

## 2. TF-IDF Features

In [ ]:
# Extract TF-IDF features
tfidf_extractor = SequenceFeatureExtractor(
    ngram_range=(1, 3),
    max_features=5000,
    use_tfidf=True,
)

X_train_tfidf = tfidf_extractor.fit_transform(train_sequences)
X_val_tfidf = tfidf_extractor.transform(val_sequences)

print(f"TF-IDF train shape: {X_train_tfidf.shape}")
print(f"TF-IDF val shape: {X_val_tfidf.shape}")

In [ ]:
# Check feature sparsity
sparsity = (X_train_tfidf == 0).sum() / X_train_tfidf.size * 100
print(f"Feature matrix sparsity: {sparsity:.2f}%")

## 3. Statistical Features

In [ ]:
# Extract statistical features
stats_extractor = StatisticalFeatureExtractor(
    include_transitions=True,
    top_k_actions=50,
)

X_train_stats = stats_extractor.fit_transform(train_sequences)
X_val_stats = stats_extractor.transform(val_sequences)

print(f"Statistical train shape: {X_train_stats.shape}")
print(f"Statistical val shape: {X_val_stats.shape}")

In [ ]:
# Feature names
stat_feature_names = stats_extractor.get_feature_names()
print(f"Number of statistical features: {len(stat_feature_names)}")
print(f"Sample feature names: {stat_feature_names[:10]}")

## 4. Complete Feature Pipeline

In [ ]:
# Create complete pipeline
pipeline = FeaturePipeline(
    use_tfidf=True,
    use_ngrams=True,
    use_statistics=True,
    ngram_range=(1, 3),
    max_features=5000,
    top_k_actions=50,
)

X_train_features = pipeline.fit_transform(train_sequences)
X_val_features = pipeline.transform(val_sequences)

print(f"Final train features shape: {X_train_features.shape}")
print(f"Final val features shape: {X_val_features.shape}")

In [ ]:
# Save pipeline
pipeline.save('../outputs/models/feature_pipeline.pkl')
print("Feature pipeline saved!")

## 5. Target Encoding

In [ ]:
# Encode targets
target_encoder = TargetEncoder(DataLoader.TARGET_COLS)
y_train = target_encoder.fit_transform(Y_train)
y_val = target_encoder.transform(Y_val)

print(f"Encoded y_train shape: {y_train.shape}")
print(f"Encoded y_val shape: {y_val.shape}")
print(f"\nNumber of classes per target: {target_encoder.num_classes}")

## Summary

### Features Created

| Feature Group | Extractor | # Features | Used by |
|---------------|-----------|-----------|---------|
| TF-IDF (ngram 1?3) | `SequenceFeatureExtractor` | 5,000 | XGBoost, LightGBM |
| N-gram counts | `SequenceFeatureExtractor` | 300 | XGBoost, LightGBM |
| Statistical | `StatisticalFeatureExtractor` | 118 | XGBoost, LightGBM |
| Histogram (254 bins) + last-K | `HistogramFeatureExtractor` | 1,524 | XGBoost, LightGBM |
| **Total ML features** | `FeaturePipeline` | **6,942** | XGBoost, LightGBM |
| Padded sequences (dense vocab) | `SequencePreprocessor` | 64 (length) | LSTM, Transformer |

### Data Leakage Verification

- `SequencePreprocessor.fit()` called on `train_sequences` only
- `FeaturePipeline.fit()` called on `train_sequences` only
- `TargetEncoder.fit()` called on `Y_train` only
- Dense vocab mapping built from train tokens only (254 unique actions ? indices 1?254)
- Val/test sequences transformed using pre-fitted preprocessors (no new information leaked)

### Next Steps (Notebook 03)

- Train XGBoost and LightGBM (ML) using `FeaturePipeline` features
- Train LSTM (DL) and Transformer using padded sequences
- Build Ensemble (weighted soft-voting)
- All models trained on `X_train` with **internal early-stop fold only** ? `Y_val` is held out
- Evaluate all models on `X_val / Y_val` (held-out) and report Exact-Match Accuracy